# PAN12 and Phishing Dataset Study

This notebook explores the cleaned PAN12 chat files and the phishing URL dataset in `data/processed/` and suggests a practical cleaning procedure for model training.

Focus areas:
- label balance
- text length distribution
- missing values and duplicates
- sample inspection
- a cleaning procedure you can apply consistently

In [8]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 140)

BASE_DIR = Path('..').resolve()
DATA_DIR = BASE_DIR / 'processed'
TRAIN_PATH = DATA_DIR / 'pan12_train_clean.csv'
TEST_PATH = DATA_DIR / 'pan12_test_clean.csv'

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df['split'] = 'train'
test_df['split'] = 'test'
df = pd.concat([train_df, test_df], ignore_index=True)

print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)
print('Combined   :', df.shape)
print('Columns    :', list(df.columns))

Train shape: (735122, 3)
Test shape : (1668249, 3)
Combined   : (2403371, 3)
Columns    : ['text', 'label', 'split']


In [9]:
label_counts = df['label'].value_counts().sort_index()
label_pct = (label_counts / len(df) * 100).round(2)
summary = pd.DataFrame({'count': label_counts, 'percent': label_pct})
summary.index = summary.index.map({0: 'normal (0)', 1: 'predator (1)'})
summary

,count,percent
label,,
normal (0),2204323,91.72
predator (1),199048,8.28


In [10]:
# Basic data quality checks
quality = pd.DataFrame({
    'null_text': [int(df['text'].isna().sum())],
    'empty_text': [int((df['text'].astype(str).str.strip() == '').sum())],
    'duplicate_rows': [int(df.duplicated(subset=['text', 'label']).sum())],
    'duplicate_texts': [int(df.duplicated(subset=['text']).sum())],
})
quality

,null_text,empty_text,duplicate_rows,duplicate_texts
0,0,0,693120,704036


In [11]:
# Text length statistics
text_series = df['text'].astype(str).str.strip()
df['char_len'] = text_series.str.len()
df['word_len'] = text_series.str.split().map(len)

length_stats = df.groupby('label')[['char_len', 'word_len']].agg(['count', 'mean', 'median', 'min', 'max'])
length_stats

char_len                               word_len                            
         count       mean median min     max    count      mean median min    max
label                                                                            
0      2204323  43.051181   23.0   1  413250  2204323  7.129996    4.0   1  74292
1       199048  19.192732   14.0   1     787   199048  4.503386    3.0   1    165

In [12]:
# Length quantiles by class
quantiles = df.groupby('label')['word_len'].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).unstack()
quantiles.index = quantiles.index.map({0: 'normal (0)', 1: 'predator (1)'})
quantiles

,0.10,0.25,0.50,0.75,0.90
label,,,,,
normal (0),1.0,2.0,4.0,9.0,16.0
predator (1),1.0,1.0,3.0,6.0,9.0


In [13]:
# Example rows from each class
display(df[df['label'] == 1][['split', 'label', 'text']].head(5))
display(df[df['label'] == 0][['split', 'label', 'text']].head(5))

,split,label,text
451,train,1,hey!!
452,train,1,hi!
453,train,1,how are you feeling?
454,train,1,a little better what are u doing?
455,train,1,i was just listening to some music


,split,label,text
0,train,0,Hola.
1,train,0,hi.
2,train,0,whats up?
3,train,0,not a ton.
4,train,0,you?


In [14]:
# Phishing URL dataset study
PHISHING_PATH = DATA_DIR / 'phishing_urls.csv'
phishing_df = pd.read_csv(PHISHING_PATH)

# Keep only the URL text and label columns.
# label: 1 = phishing, 0 = not phishing
phishing_df = phishing_df[['text', 'label']].copy()
phishing_df['label'] = phishing_df['label'].astype(int)
phishing_df['text'] = phishing_df['text'].astype(str).str.strip()

print('Phishing shape:', phishing_df.shape)
print('Phishing columns:', list(phishing_df.columns))
print()
print('Label distribution:')
print(phishing_df['label'].value_counts().sort_index())
print()

phishing_text = phishing_df['text']
phishing_df['char_len'] = phishing_text.str.len()
phishing_df['word_len'] = phishing_text.str.split().map(len)

phishing_quality = pd.DataFrame({
    'null_text': [int(phishing_df['text'].isna().sum())],
    'empty_text': [int((phishing_text == '').sum())],
    'duplicate_rows': [int(phishing_df.duplicated(subset=['text', 'label']).sum())],
    'duplicate_texts': [int(phishing_df.duplicated(subset=['text']).sum())],
})

print('\nQuality checks:')
display(phishing_quality)

print('\nLength statistics by label:')
display(phishing_df.groupby('label')[['char_len', 'word_len']].agg(['count', 'mean', 'median', 'min', 'max']))

print('\nSample phishing rows:')
display(phishing_df[['text', 'label']].head(5))

Phishing shape: (77677, 2)
Phishing columns: ['text', 'label']

Label distribution:
label
0    44975
1    32702
Name: count, dtype: int64


Quality checks:


,null_text,empty_text,duplicate_rows,duplicate_texts
0,0,1,238,238



Length statistics by label:


char_len                                   word_len                                
         count         mean median min       max    count        mean median min      max
label                                                                                    
0        44975  8718.441134   93.0   2  17036692    44975  632.415075    5.0   1  3527576
1        32702  3216.070760   64.0   0    129633    32702  216.686716    1.0   0    11625


Sample phishing rows:


,text,label
0,<!doctypehtml><html lang=en xml:lang=en xmlns=http://www.w3.org/1999/xhtml><title>\n Clipper Servicing and Repair ...,0
1,http://online0mgeving.ga/triodos/,1
2,metronews.ca/webapp/Login.aspx?logout=true&rurl=/toronto/sports/article/1022417--impact-name-mauro-biello-assistant-...,0
3,https://rarkuntem.co.jp.fpjiehk.cn/index1.php,1
4,freebase.com/view/m/0ct05qn,0


## Suggested Cleaning Procedure

Use the following cleaning steps before training a model:

1. Keep only the final schema you need: `text` and `label`.
2. For phishing URLs, store the URL string in `text` and use `label` where `1 = phishing` and `0 = not phishing`.
3. Strip leading/trailing whitespace and collapse repeated whitespace into a single space.
4. Remove empty rows after normalization.
5. Drop exact duplicate `(text, label)` rows.
6. For PAN12 chat messages, optionally mask URLs, emails, and user handles with placeholders such as `<URL>`, `<EMAIL>`, and `<USER>`.
7. For phishing URLs, keep the URL string intact for URL-based models, or normalize it consistently if you are training a classical text model.
8. Lowercasing is optional for classical models; for transformer models, usually keep the original casing unless you have a reason not to.
9. Keep short messages and short URLs because important signal often lives in compact text.
10. Split by conversation for PAN12 so that messages from the same conversation do not leak across train and validation sets.
11. Save the cleaned dataset separately from the raw source data so you can reproduce experiments.

## Dataset-Specific Notes

### PAN12 Chat Data
- Preserve conversation boundaries during splitting.
- Normalize chat artifacts like repeated punctuation and obvious placeholders if needed.
- Avoid aggressive token removal that could erase grooming cues.

### Phishing URL Data
- Keep only the URL text and the binary label column.
- Remove duplicate URLs and obviously malformed rows.
- Consider stripping query parameters only if your downstream task is about domain-level classification rather than exact URL matching.
- If you train on raw URLs, keep character-level signal such as `-`, `.`, `/`, and `?` because those often matter.

## Practical Recommendation

For this project, the safest default is a light cleaning pipeline: whitespace normalization, duplicate removal, and optional masking/normalization only where it preserves the signal you care about. That keeps the grooming and phishing patterns intact while removing noise.